# 🚀 Day 25 — Ensemble Learning
## Bank Customer Subscription Prediction

**Objective:** Build a practical machine learning classification project using multiple ensemble techniques to predict whether a bank customer will subscribe to a term deposit.

**Dataset:** UCI Bank Marketing — `bank-full.csv`

**Models:** Logistic Regression, Random Forest, Gradient Boosting, and Soft Voting Ensemble.

## 📌 Important — Google Colab Dataset Upload

Run the next cell first. It will open a file picker. Upload the real **`bank-full.csv`** file downloaded from the UCI Bank Marketing dataset.

The CSV uses `;` as its separator.

In [ ]:
from google.colab import files

uploaded = files.upload()

print('Uploaded files:', list(uploaded.keys()))

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    ConfusionMatrixDisplay, roc_curve
)

RANDOM_STATE = 42
print('Libraries imported successfully!')

## 1. Load the Dataset

In [ ]:
DATA_PATH = 'bank-full.csv'
df = pd.read_csv(DATA_PATH, sep=';')

print('Dataset shape:', df.shape)
display(df.head())

## 2. Understand the Dataset

In [ ]:
print('Columns:')
print(df.columns.tolist())

print('\nData types:')
display(df.dtypes)

print('\nMissing values:')
display(df.isnull().sum())

print('\nDuplicate rows:', df.duplicated().sum())

In [ ]:
print('Target distribution:')
display(df['y'].value_counts())

target_counts = df['y'].value_counts()
plt.figure(figsize=(6,4))
plt.bar(target_counts.index, target_counts.values)
plt.title('Term Deposit Subscription Distribution')
plt.xlabel('Subscribed')
plt.ylabel('Customers')
plt.show()

## 3. Data Cleaning and Target Encoding

In [ ]:
df = df.drop_duplicates().copy()
df['y'] = df['y'].map({'no': 0, 'yes': 1})

print('Shape after removing duplicates:', df.shape)
print('\nEncoded target distribution:')
display(df['y'].value_counts())

## 4. Feature Engineering

In [ ]:
df['age_group'] = pd.cut(
    df['age'],
    bins=[0, 25, 35, 45, 55, 100],
    labels=['Young', 'Adult', 'Mid_Age', 'Senior', 'Elder']
)

df['balance_category'] = pd.cut(
    df['balance'],
    bins=[-np.inf, 0, 1000, 5000, np.inf],
    labels=['Negative', 'Low', 'Medium', 'High']
)

df['campaign_intensity'] = pd.cut(
    df['campaign'],
    bins=[0, 1, 3, 5, np.inf],
    labels=['Low', 'Medium', 'High', 'Very_High']
)

display(df.head())

## 5. Business Exploration

In [ ]:
subscription_rate = df['y'].mean() * 100
print(f'Overall subscription rate: {subscription_rate:.2f}%')

job_subscription = (
    df.groupby('job', observed=False)['y']
      .mean()
      .sort_values(ascending=False) * 100
)

print('\nSubscription rate by job:')
display(job_subscription.round(2).to_frame('Subscription Rate (%)'))

In [ ]:
plt.figure(figsize=(10,5))
plt.bar(job_subscription.index, job_subscription.values)
plt.xticks(rotation=45, ha='right')
plt.ylabel('Subscription Rate (%)')
plt.xlabel('Job')
plt.title('Subscription Rate by Job')
plt.tight_layout()
plt.show()

## 6. Prepare Features and Target

In [ ]:
X = df.drop(columns=['y'])
y = df['y']

numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object', 'category']).columns.tolist()

print('Numeric features:', numeric_features)
print('\nCategorical features:', categorical_features)

## 7. Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print('Training shape:', X_train.shape)
print('Testing shape:', X_test.shape)

## 8. Preprocessing Pipeline

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
    ]
)

print('Preprocessing pipeline created.')

## 9. Define Individual Models

In [ ]:
logistic_regression = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(
        max_iter=2000,
        class_weight='balanced',
        random_state=RANDOM_STATE
    ))
])

random_forest = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestClassifier(
        n_estimators=200,
        max_depth=15,
        min_samples_split=5,
        min_samples_leaf=2,
        class_weight='balanced',
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

gradient_boosting = Pipeline([
    ('preprocessor', preprocessor),
    ('model', GradientBoostingClassifier(
        n_estimators=150,
        learning_rate=0.05,
        max_depth=3,
        random_state=RANDOM_STATE
    ))
])

print('Individual models created.')

## 10. Create Soft Voting Ensemble

In [ ]:
voting_ensemble = VotingClassifier(
    estimators=[
        ('lr', logistic_regression),
        ('rf', random_forest),
        ('gb', gradient_boosting)
    ],
    voting='soft'
)

print('Soft Voting Ensemble created.')

## 11. Train and Evaluate All Models

In [ ]:
models = {
    'Logistic Regression': logistic_regression,
    'Random Forest': random_forest,
    'Gradient Boosting': gradient_boosting,
    'Voting Ensemble': voting_ensemble
}

results = []

for model_name, model in models.items():
    print(f'\nTraining {model_name}...')
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    results.append({
        'Model': model_name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, zero_division=0),
        'Recall': recall_score(y_test, y_pred, zero_division=0),
        'F1 Score': f1_score(y_test, y_pred, zero_division=0),
        'ROC-AUC': roc_auc_score(y_test, y_prob)
    })

results_df = pd.DataFrame(results).sort_values('ROC-AUC', ascending=False).reset_index(drop=True)

display(results_df.round(4))

## 12. Model Comparison

In [ ]:
metrics_to_plot = ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'ROC-AUC']

results_plot = results_df.set_index('Model')[metrics_to_plot]
results_plot.plot(kind='bar', figsize=(12,6))
plt.title('Ensemble Learning Model Comparison')
plt.ylabel('Score')
plt.ylim(0, 1)
plt.xticks(rotation=20)
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

## 13. Faster Cross-Validation

To keep the Colab notebook practical, 3-fold stratified cross-validation is applied to Random Forest and Gradient Boosting rather than running expensive nested cross-validation on every ensemble model.

In [ ]:
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

cv_models = {
    'Random Forest': random_forest,
    'Gradient Boosting': gradient_boosting
}

cv_results = []

for model_name, model in cv_models.items():
    print(f'Running 3-fold CV for {model_name}...')
    scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring='roc_auc',
        n_jobs=-1
    )

    cv_results.append({
        'Model': model_name,
        'Mean ROC-AUC': scores.mean(),
        'Std ROC-AUC': scores.std()
    })

cv_results_df = pd.DataFrame(cv_results)
display(cv_results_df.round(4))

## 14. Select the Best Model

In [ ]:
best_model_name = results_df.loc[0, 'Model']
best_model = models[best_model_name]

print('Best model based on holdout ROC-AUC:', best_model_name)
print(f"ROC-AUC: {results_df.loc[0, 'ROC-AUC']:.4f}")

## 15. Confusion Matrix

In [ ]:
best_predictions = best_model.predict(X_test)

cm = confusion_matrix(y_test, best_predictions)
print('Confusion Matrix:')
print(cm)

ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=['No Subscription', 'Subscription']
).plot()
plt.title(f'Confusion Matrix — {best_model_name}')
plt.show()

## 16. Classification Report

In [ ]:
print(classification_report(
    y_test,
    best_predictions,
    target_names=['No Subscription', 'Subscription'],
    zero_division=0
))

## 17. ROC Curve Comparison

In [ ]:
plt.figure(figsize=(9,7))

for model_name, model in models.items():
    y_prob = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    plt.plot(fpr, tpr, label=f'{model_name} (AUC={auc:.3f})')

plt.plot([0, 1], [0, 1], linestyle='--', label='Random Classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve Comparison')
plt.legend()
plt.grid(True)
plt.show()

## 18. Random Forest Feature Importance

In [ ]:
rf_model = random_forest.named_steps['model']
rf_preprocessor = random_forest.named_steps['preprocessor']

feature_names = rf_preprocessor.get_feature_names_out()
importances = rf_model.feature_importances_

feature_importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values('Importance', ascending=False).reset_index(drop=True)

display(feature_importance_df.head(20).round(6))

In [ ]:
top_features = feature_importance_df.head(15).sort_values('Importance')

plt.figure(figsize=(10,7))
plt.barh(top_features['Feature'], top_features['Importance'])
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.title('Top Random Forest Features')
plt.tight_layout()
plt.show()

## 19. Business Interpretation

- **Ensemble learning** combines multiple models to improve predictive performance and robustness.
- **Random Forest** uses bagging and many decision trees to reduce variance.
- **Gradient Boosting** builds trees sequentially, with each new tree improving previous errors.
- **Soft Voting** combines predicted probabilities from multiple classifiers.
- The model can help a bank **prioritize customers who are more likely to subscribe**, supporting more targeted marketing campaigns.
- ROC-AUC is useful here because the target classes are not perfectly balanced and ranking customers by probability can be valuable.

## 20. Save Project Results

In [ ]:
results_df.to_csv('Day25_Ensemble_Model_Results.csv', index=False)
cv_results_df.to_csv('Day25_Cross_Validation_Results.csv', index=False)
feature_importance_df.to_csv('Day25_RandomForest_Feature_Importance.csv', index=False)

print('Saved files:')
print('1. Day25_Ensemble_Model_Results.csv')
print('2. Day25_Cross_Validation_Results.csv')
print('3. Day25_RandomForest_Feature_Importance.csv')

## 🎯 Key Learnings

1. Ensemble learning combines multiple models to obtain stronger predictions.
2. Bagging mainly reduces variance, while boosting focuses on correcting previous errors.
3. Voting ensembles can combine complementary models.
4. Cross-validation gives a more reliable estimate of model performance.
5. ROC-AUC, precision, recall, and F1 provide more information than accuracy alone.
6. Feature importance can help connect ML predictions with business decisions.

**Day 25/30 — Ensemble Learning completed! 🚀**